In [0]:
Bronze_Schema_Path = "assesment_pipline.bronze"


In [0]:
# parameter widget 
dbutils.widgets.text("input_file_path", "")
dbutils.widgets.text("batch_name", "")
dbutils.widgets.text("source_name", "")
dbutils.widgets.text("raw_records_table_name", "")
dbutils.widgets.text("valid_records_table_name", "")
dbutils.widgets.text("invalid_records_table_name", "")

#  Fetch parameter values
input_file_path = dbutils.widgets.get("input_file_path")
batch_name = dbutils.widgets.get("batch_name")
source_name = dbutils.widgets.get("source_name")
raw_records_table_name = dbutils.widgets.get("raw_records_table_name")
valid_records_table_name = dbutils.widgets.get("valid_records_table_name")
invalid_records_table_name = dbutils.widgets.get("invalid_records_table_name")

In [0]:
input_df=spark.read.csv(input_file_path,header=True,inferSchema=True)
display(input_df)

order_id,order_date,customer_id,product_id,product_name,category,shipping_city,quantity,unit_price,discount,order_status
ORD0001,2026-10-07,C001,P001,Product 2,GROCERY,Delhi,4,375.0,20.0,DELIVERED
ORD0002,2026-10-08,C002,P002,Product 3,BEVERAGES,Coimbatore,5,550.0,40.0,CANCELLED
ORD0003,2026-10-09,C003,P003,Product 4,HOUSEHOLD,Bengaluru,6,725.0,60.0,PLACED
ORD0004,2026-10-10,C004,P004,Product 5,GROCERY,Chennai,7,900.0,0.0,DELIVERED
ORD0101,2026-10-23,C001,P001,Product 6,BEVERAGES,Hyderabad,2,1075.0,20.0,DELIVERED
ORD0102,2026-10-24,C002,P002,Product 7,HOUSEHOLD,Pune,3,1250.0,40.0,CANCELLED
ORD0103,2026-10-25,C003,P003,Product 8,GROCERY,Mumbai,4,1425.0,60.0,PLACED
ORD0104,2026-10-26,C004,P004,Product 1,BEVERAGES,Kochi,5,200.0,0.0,DELIVERED
ORD0105,2026-10-27,C005,P005,Product 2,HOUSEHOLD,Delhi,1,375.0,20.0,CANCELLED
ORD0106,2026-10-28,C006,P006,Product 3,GROCERY,Coimbatore,2,550.0,40.0,PLACED


In [0]:
from pyspark.sql.functions import *
bronze_df=input_df.withColumn("batch_id",lit(batch_name)).withColumn("source_file",lit(source_name)).withColumn("ingestion_timestamp",lit(current_timestamp()))
display(bronze_df)
bronze_df.write.mode("overwrite").format("delta").saveAsTable(f"{Bronze_Schema_Path}.{raw_records_table_name}")

order_id,order_date,customer_id,product_id,product_name,category,shipping_city,quantity,unit_price,discount,order_status,batch_id,source_file,ingestion_timestamp
ORD0001,2026-10-07,C001,P001,Product 2,GROCERY,Delhi,4,375.0,20.0,DELIVERED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:03.721Z
ORD0002,2026-10-08,C002,P002,Product 3,BEVERAGES,Coimbatore,5,550.0,40.0,CANCELLED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:03.721Z
ORD0003,2026-10-09,C003,P003,Product 4,HOUSEHOLD,Bengaluru,6,725.0,60.0,PLACED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:03.721Z
ORD0004,2026-10-10,C004,P004,Product 5,GROCERY,Chennai,7,900.0,0.0,DELIVERED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:03.721Z
ORD0101,2026-10-23,C001,P001,Product 6,BEVERAGES,Hyderabad,2,1075.0,20.0,DELIVERED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:03.721Z
ORD0102,2026-10-24,C002,P002,Product 7,HOUSEHOLD,Pune,3,1250.0,40.0,CANCELLED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:03.721Z
ORD0103,2026-10-25,C003,P003,Product 8,GROCERY,Mumbai,4,1425.0,60.0,PLACED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:03.721Z
ORD0104,2026-10-26,C004,P004,Product 1,BEVERAGES,Kochi,5,200.0,0.0,DELIVERED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:03.721Z
ORD0105,2026-10-27,C005,P005,Product 2,HOUSEHOLD,Delhi,1,375.0,20.0,CANCELLED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:03.721Z
ORD0106,2026-10-28,C006,P006,Product 3,GROCERY,Coimbatore,2,550.0,40.0,PLACED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:03.721Z


In [0]:
bronze_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- shipping_city: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- order_status: string (nullable = true)
 |-- batch_id: string (nullable = false)
 |-- source_file: string (nullable = false)
 |-- ingestion_timestamp: timestamp (nullable = false)



In [0]:
from pyspark.sql.functions import *

order_id_condition = (
    col("order_id").isNull() |
    (trim(col("order_id")) == "")
)

order_date_condition = col("order_date").isNull()

customer_id_condition = (
    col("customer_id").isNull() |
    (trim(col("customer_id")) == "")
)

product_id_condition = (
    col("product_id").isNull() |
    (trim(col("product_id")) == "")
)

product_name_condition = (
    col("product_name").isNull() |
    (trim(col("product_name")) == "")
)

category_condition = (
    col("category").isNull() |
    (trim(col("category")) == "")
)

shipping_city_condition = (
    col("shipping_city").isNull() |
    (trim(col("shipping_city")) == "")
)

quantity_condition = (
    col("quantity").isNull() |
    (col("quantity") <= 0)
)

unit_price_condition = (
    col("unit_price").isNull() |
    (col("unit_price") <= 0)
)

discount_condition = (
    col("discount").isNull() |
    (col("discount") < 0)
)

order_status_condition = (
    col("order_status").isNull() |
    (~upper(trim(col("order_status"))).isin(
        "PLACED", "SHIPPED","DELIVERED", "CANCELLED"
    ))
)

invalied_condition = (
    order_id_condition |
    order_date_condition |
    customer_id_condition |
    product_id_condition |
    product_name_condition |
    category_condition |
    shipping_city_condition |
    quantity_condition |
    unit_price_condition |
    discount_condition |
    order_status_condition 
)

In [0]:

valid_records = bronze_df.filter(
    ~invalied_condition
)

valid_records_count = valid_records.count()

display(valid_records.orderBy("order_id"))

order_id,order_date,customer_id,product_id,product_name,category,shipping_city,quantity,unit_price,discount,order_status,batch_id,source_file,ingestion_timestamp
ORD0001,2026-10-07,C001,P001,Product 2,GROCERY,Delhi,4,375.0,20.0,DELIVERED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:07.626Z
ORD0002,2026-10-08,C002,P002,Product 3,BEVERAGES,Coimbatore,5,550.0,40.0,CANCELLED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:07.626Z
ORD0003,2026-10-09,C003,P003,Product 4,HOUSEHOLD,Bengaluru,6,725.0,60.0,PLACED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:07.626Z
ORD0004,2026-10-10,C004,P004,Product 5,GROCERY,Chennai,7,900.0,0.0,DELIVERED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:07.626Z
ORD0101,2026-10-23,C001,P001,Product 6,BEVERAGES,Hyderabad,2,1075.0,20.0,DELIVERED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:07.626Z
ORD0102,2026-10-24,C002,P002,Product 7,HOUSEHOLD,Pune,3,1250.0,40.0,CANCELLED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:07.626Z
ORD0103,2026-10-25,C003,P003,Product 8,GROCERY,Mumbai,4,1425.0,60.0,PLACED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:07.626Z
ORD0104,2026-10-26,C004,P004,Product 1,BEVERAGES,Kochi,5,200.0,0.0,DELIVERED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:07.626Z
ORD0105,2026-10-27,C005,P005,Product 2,HOUSEHOLD,Delhi,1,375.0,20.0,CANCELLED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:07.626Z
ORD0106,2026-10-28,C006,P006,Product 3,GROCERY,Coimbatore,2,550.0,40.0,PLACED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:07.626Z


In [0]:

invalied_records = bronze_df.filter(
    invalied_condition
)

invalied_records = invalied_records.withColumn(
    "Rejected_reason",
    concat_ws(",",
        when(order_id_condition, "order_id is null or blank"),
        when(order_date_condition, "order_date is null"),
        when(customer_id_condition, "customer_id is null or blank"),
        when(product_id_condition, "product_id is null or blank"),
        when(product_name_condition, "product_name is null or blank"),
        when(category_condition, "category is null or blank"),
        when(shipping_city_condition, "shipping_city is null or blank"),
        when(quantity_condition, "quantity is null or less than or equal to zero"),
        when(unit_price_condition, "unit_price is null or less than or equal to zero"),
        when(discount_condition, "discount is null or less than zero"),
        when(order_status_condition, "invalid order_status")
    )
)
# .drop("_id_count")

invalied_records_count = invalied_records.count()

display(invalied_records)



order_id,order_date,customer_id,product_id,product_name,category,shipping_city,quantity,unit_price,discount,order_status,batch_id,source_file,ingestion_timestamp,Rejected_reason


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window_spec = Window.partitionBy("order_id").orderBy(col("order_date").desc())

valid_records_with_rn = valid_records.withColumn(
    "row_num",
    row_number().over(window_spec)
)

valid_records = valid_records_with_rn.filter(
    col("row_num") == 1
).drop("row_num")

duplicate_records = valid_records_with_rn.filter(
    col("row_num") > 1
).drop("row_num")

# Adding rejection reason
duplicate_records = duplicate_records.withColumn(
    "Rejected_reason",
    lit("Duplicate order_id")
)

invalied_records = invalied_records.unionByName(
    duplicate_records,
    allowMissingColumns=True
)

display(valid_records)
valid_records.write.mode("overwrite").format("delta").saveAsTable(
    f"{Bronze_Schema_Path}.{valid_records_table_name}"
)

display(invalied_records)
invalied_records.write.mode("overwrite").format("delta").saveAsTable(
    f"{Bronze_Schema_Path}.{invalid_records_table_name}"
)

order_id,order_date,customer_id,product_id,product_name,category,shipping_city,quantity,unit_price,discount,order_status,batch_id,source_file,ingestion_timestamp
ORD0001,2026-10-07,C001,P001,Product 2,GROCERY,Delhi,4,375.0,20.0,DELIVERED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:09.145Z
ORD0002,2026-10-08,C002,P002,Product 3,BEVERAGES,Coimbatore,5,550.0,40.0,CANCELLED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:09.145Z
ORD0003,2026-10-09,C003,P003,Product 4,HOUSEHOLD,Bengaluru,6,725.0,60.0,PLACED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:09.145Z
ORD0004,2026-10-10,C004,P004,Product 5,GROCERY,Chennai,7,900.0,0.0,DELIVERED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:09.145Z
ORD0101,2026-10-23,C001,P001,Product 6,BEVERAGES,Hyderabad,2,1075.0,20.0,DELIVERED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:09.145Z
ORD0102,2026-10-24,C002,P002,Product 7,HOUSEHOLD,Pune,3,1250.0,40.0,CANCELLED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:09.145Z
ORD0103,2026-10-25,C003,P003,Product 8,GROCERY,Mumbai,4,1425.0,60.0,PLACED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:09.145Z
ORD0104,2026-10-26,C004,P004,Product 1,BEVERAGES,Kochi,5,200.0,0.0,DELIVERED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:09.145Z
ORD0105,2026-10-27,C005,P005,Product 2,HOUSEHOLD,Delhi,1,375.0,20.0,CANCELLED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:09.145Z
ORD0106,2026-10-28,C006,P006,Product 3,GROCERY,Coimbatore,2,550.0,40.0,PLACED,BATCH_002,online_shopping_incremental.csv,2026-11-23T07:18:09.145Z


order_id,order_date,customer_id,product_id,product_name,category,shipping_city,quantity,unit_price,discount,order_status,batch_id,source_file,ingestion_timestamp,Rejected_reason


In [0]:
source_record_count = bronze_df.count()

expected_silver_count = (
    source_record_count - invalied_records_count
)

dbutils.jobs.taskValues.set(
    key="expected_silver_count",
    value=expected_silver_count
)

print("Source Count:", source_record_count)
print("Valid Count:", valid_records_count)
print("Invalid Count:", invalied_records_count)
print("Expected Silver Count:", expected_silver_count)

Source Count: 12
Valid Count: 12
Invalid Count: 0
Expected Silver Count: 12
